# **Загружаем все синсеты и senses (N + A + V)**

In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
from collections import defaultdict
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

RWN_DIR = Path('/content/drive/MyDrive/SFU 4/VKR/5. ruwordnet/xmls')

# ── Шаг 1: Загружаем все синсеты и senses (N + A + V) ─────────────
print('Загружаем RuWordNet...')

synsets = {}      # id → {title, definition, pos, words}
word_to_synsets = defaultdict(list)  # слово → [synset_id]

for pos in ['N', 'A', 'V']:
    # Синсеты
    tree = ET.parse(RWN_DIR / f'synsets.{pos}.xml')
    for synset in tree.getroot():
        sid = synset.attrib['id']
        synsets[sid] = {
            'id': sid,
            'title': synset.attrib.get('ruthes_name', ''),
            'definition': synset.attrib.get('definition', ''),
            'pos': pos,
            'words': []
        }

    # Senses — привязываем слова к синсетам
    tree = ET.parse(RWN_DIR / f'senses.{pos}.xml')
    for sense in tree.getroot():
        sid = sense.attrib.get('synset_id', '')
        word = sense.attrib.get('lemma', '').lower()
        if sid in synsets and word:
            synsets[sid]['words'].append(word)
            word_to_synsets[word].append(sid)

print(f'Синсетов загружено: {len(synsets)}')
print(f'Уникальных слов:    {len(word_to_synsets)}')

# Тест
for test_word in ['депрессия', 'злость', 'мучиться', 'страх']:
    sids = word_to_synsets.get(test_word, [])
    print(f'\n«{test_word}» → {len(sids)} синсетов:')
    for sid in sids[:2]:
        s = synsets[sid]
        print(f'  [{s["pos"]}] {s["title"]}: {s["words"][:6]}')

Загружаем RuWordNet...
Синсетов загружено: 59905
Уникальных слов:    133250

«депрессия» → 3 синсетов:
  [N] ЭКОНОМИЧЕСКАЯ ДЕПРЕССИЯ: ['обвал экономика', 'экономический депрессия', 'обвал в экономика', 'депрессия']
  [N] ДЕПРЕССИВНОЕ РАССТРОЙСТВО: ['депрессия', 'нервный депрессия', 'депрессивный состояние', 'депрессивность', 'депрессивный расстройство']

«злость» → 1 синсетов:
  [N] ЗЛОБА: ['злость', 'ожесточение', 'злобность', 'озлобление', 'ожесточенность', 'злоба']

«мучиться» → 1 синсетов:
  [V] СТРАДАТЬ, ИСПЫТЫВАТЬ СТРАДАНИЯ: ['помучиться', 'измучиться', 'исстрадаться', 'испытывать мука', 'замучиться', 'испытывать мучение']

«страх» → 1 синсетов:
  [N] СТРАХ, БОЯЗНЬ: ['боязнь', 'страх', 'трепет']


# **Загружаем слова из semantic_fields_VKR.xlsx**

In [2]:
import pandas as pd
import openpyxl

# ── Шаг 2: Загружаем слова из semantic_fields_VKR.xlsx ────────────
EXCEL_PATH = '/content/drive/MyDrive/SFU 4/VKR/4. word2vec/semantic_fields_VKR.xlsx'
wb_in = openpyxl.load_workbook(EXCEL_PATH)
ws_in = wb_in.active

records = []
current_keyword = None

for row in ws_in.iter_rows(values_only=True):
    # Строка с ключевым словом
    if row[0] and isinstance(row[0], str) and row[0].startswith('Слово:'):
        current_keyword = row[0].replace('Слово: «', '').replace('»', '').strip()
        continue
    # Пропускаем заголовки
    if row[0] in ['depression', 'Слово', None]:
        continue
    # Строки с данными — 6 колонок: слово_dep, score_dep, слово_anx, score_anx, слово_neu, score_neu
    if current_keyword and row[0] and isinstance(row[0], str):
        for col_idx, class_label in [(0, 'depression'), (2, 'anxiety'), (4, 'neutral')]:
            word  = row[col_idx]
            score = row[col_idx + 1] if col_idx + 1 < len(row) else None
            if word and isinstance(word, str) and word not in ['—', '']:
                records.append({
                    'keyword':  current_keyword,
                    'class':    class_label,
                    'neighbor': word.strip().lower(),
                    'score':    score
                })

df_words = pd.DataFrame(records)
print(f'Загружено пар: {len(df_words)}')
print(f'Ключевых слов: {df_words["keyword"].nunique()}')
print(df_words["keyword"].unique())
print()
print(df_words.head(10))

Загружено пар: 260
Ключевых слов: 9
['депрессия' 'тревога' 'страх' 'одиночество' 'помощь' 'лечение' 'чувство'
 'жизнь' 'плохо']

     keyword       class        neighbor  score
0  депрессия  depression       исключать  0.635
1  депрессия     anxiety          низкий  0.716
2  депрессия     neutral          злость  0.724
3  депрессия  depression        динамика  0.628
4  депрессия     anxiety      показатель  0.711
5  депрессия     neutral   самообвинение  0.694
6  депрессия  depression  второстепенный  0.620
7  депрессия     anxiety        опросник  0.710
8  депрессия     neutral    руминировать  0.689
9  депрессия  depression     униполярный  0.619


# **Для каждого соседа ищем синсеты в RuWordNet**

In [3]:
from collections import defaultdict

# ── Шаг 3: Для каждого соседа ищем синсеты в RuWordNet ────────────

def get_synset_info(word):
    """Возвращает список синсетов для слова"""
    sids = word_to_synsets.get(word, [])
    result = []
    for sid in sids:
        s = synsets[sid]
        result.append({
            'synset_id': sid,
            'synset_title': s['title'],
            'pos': s['pos'],
            'synset_words': s['words']
        })
    return result

def check_same_synset(word1, word2):
    """Проверяет входят ли два слова в один синсет"""
    s1 = set(word_to_synsets.get(word1, []))
    s2 = set(word_to_synsets.get(word2, []))
    common = s1 & s2
    return list(common)

# Применяем к датафрейму
results = []
for _, row in df_words.iterrows():
    keyword   = row['keyword']
    neighbor  = row['neighbor']
    cls       = row['class']
    score     = row['score']

    # Синсеты ключевого слова
    kw_synsets = get_synset_info(keyword)
    kw_synset_ids = set(word_to_synsets.get(keyword, []))
    kw_synset_titles = [s['synset_title'] for s in kw_synsets]

    # Синсеты соседа
    nb_synsets = get_synset_info(neighbor)
    nb_synset_ids = set(word_to_synsets.get(neighbor, []))
    nb_synset_titles = [s['synset_title'] for s in nb_synsets]

    # Общие синсеты
    common_ids = kw_synset_ids & nb_synset_ids
    same_synset = len(common_ids) > 0

    # Гиперонимы — проверяем через synset_relations
    # (добавим позже если нужно)

    results.append({
        'keyword':          keyword,
        'class':            cls,
        'neighbor':         neighbor,
        'score':            score,
        'kw_in_rwn':        len(kw_synsets) > 0,
        'nb_in_rwn':        len(nb_synsets) > 0,
        'kw_synsets':       ' | '.join(kw_synset_titles),
        'nb_synsets':       ' | '.join(nb_synset_titles),
        'same_synset':      same_synset,
        'common_synsets':   ' | '.join([synsets[i]['title'] for i in common_ids]),
    })

df_result = pd.DataFrame(results)

print('══════════════════════════════════════════')
print(f'Всего пар: {len(df_result)}')
print(f'Соседей найдено в RuWordNet: {df_result["nb_in_rwn"].sum()} ({df_result["nb_in_rwn"].mean()*100:.1f}%)')
print(f'Пар в одном синсете: {df_result["same_synset"].sum()} ({df_result["same_synset"].mean()*100:.1f}%)')
print('══════════════════════════════════════════')
print()
print('── По классам ──')
for cls in ['depression', 'anxiety', 'neutral']:
    sub = df_result[df_result['class'] == cls]
    in_rwn = sub['nb_in_rwn'].mean() * 100
    same   = sub['same_synset'].mean() * 100
    print(f'{cls:<12}: в RuWordNet {in_rwn:.1f}% | в одном синсете {same:.1f}%')
print()
print('── По ключевым словам ──')
for kw in df_result['keyword'].unique():
    sub = df_result[df_result['keyword'] == kw]
    in_rwn = sub['nb_in_rwn'].mean() * 100
    same   = sub['same_synset'].mean() * 100
    print(f'«{kw}»: в RuWordNet {in_rwn:.1f}% | в одном синсете {same:.1f}%')

══════════════════════════════════════════
Всего пар: 260
Соседей найдено в RuWordNet: 213 (81.9%)
Пар в одном синсете: 3 (1.2%)
══════════════════════════════════════════

── По классам ──
depression  : в RuWordNet 84.4% | в одном синсете 2.2%
anxiety     : в RuWordNet 81.1% | в одном синсете 1.1%
neutral     : в RuWordNet 80.0% | в одном синсете 0.0%

── По ключевым словам ──
«депрессия»: в RuWordNet 73.3% | в одном синсете 0.0%
«тревога»: в RuWordNet 73.3% | в одном синсете 0.0%
«страх»: в RuWordNet 86.7% | в одном синсете 0.0%
«одиночество»: в RuWordNet 86.7% | в одном синсете 0.0%
«помощь»: в RuWordNet 96.7% | в одном синсете 3.3%
«лечение»: в RuWordNet 85.0% | в одном синсете 0.0%
«чувство»: в RuWordNet 96.7% | в одном синсете 3.3%
«жизнь»: в RuWordNet 93.3% | в одном синсете 3.3%
«плохо»: в RuWordNet 46.7% | в одном синсете 0.0%


In [4]:
# Загружаем гиперонимические отношения
hypernym_relations = {}  # synset_id → [parent_synset_id]

for pos in ['N', 'A', 'V']:
    tree = ET.parse(RWN_DIR / f'synset_relations.{pos}.xml')
    for rel in tree.getroot():
        # Смотрим структуру
        print(f'тег: {rel.tag}, атрибуты: {rel.attrib}')
        break
    break

тег: relation, атрибуты: {'name': 'hypernym', 'child_id': '113350-N', 'parent_id': '130542-N'}


In [6]:
# ── Загружаем все гиперонимические отношения ──────────────────────
print('Загружаем отношения...')
hypernyms = defaultdict(list)   # synset_id → [parent_synset_id]
hyponyms  = defaultdict(list)   # synset_id → [child_synset_id]

for pos in ['N', 'A', 'V']:
    tree = ET.parse(RWN_DIR / f'synset_relations.{pos}.xml')
    for rel in tree.getroot():
        name      = rel.attrib.get('name', '')
        child_id  = rel.attrib.get('child_id', '')
        parent_id = rel.attrib.get('parent_id', '')
        if name == 'hypernym':
            hypernyms[child_id].append(parent_id)
            hyponyms[parent_id].append(child_id)

print(f'Синсетов с гиперонимами: {len(hypernyms)}')

def get_all_hypernyms(synset_id, depth=3):
    """Получает все гиперонимы до глубины depth"""
    result = set()
    current = {synset_id}
    for _ in range(depth):
        parents = set()
        for sid in current:
            for p in hypernyms.get(sid, []):
                parents.add(p)
                result.add(p)
        current = parents
    return result

def find_common_hypernyms(word1, word2, depth=3):
    """Находит общие гиперонимы двух слов"""
    s1_ids = set(word_to_synsets.get(word1, []))
    s2_ids = set(word_to_synsets.get(word2, []))

    if not s1_ids or not s2_ids:
        return []

    # Все гиперонимы для каждого слова
    h1 = set()
    for sid in s1_ids:
        h1 |= get_all_hypernyms(sid, depth)
        h1.add(sid)

    h2 = set()
    for sid in s2_ids:
        h2 |= get_all_hypernyms(sid, depth)
        h2.add(sid)

    common = h1 & h2
    return [(sid, synsets[sid]['title']) for sid in common if sid in synsets]

# ── Применяем к датафрейму ─────────────────────────────────────────
print('Анализируем гиперонимы...')
results2 = []

for _, row in df_words.iterrows():
    keyword  = row['keyword']
    neighbor = row['neighbor']
    cls      = row['class']
    score    = row['score']

    kw_synsets_info = get_synset_info(keyword)
    nb_synsets_info = get_synset_info(neighbor)
    common_syn      = check_same_synset(keyword, neighbor)
    common_hyp      = find_common_hypernyms(keyword, neighbor, depth=3)

    results2.append({
        'keyword':           keyword,
        'class':             cls,
        'neighbor':          neighbor,
        'score':             score,
        'kw_in_rwn':         len(kw_synsets_info) > 0,
        'nb_in_rwn':         len(nb_synsets_info) > 0,
        'kw_synsets':        ' | '.join([s['synset_title'] for s in kw_synsets_info]),
        'nb_synsets':        ' | '.join([s['synset_title'] for s in nb_synsets_info]),
        'same_synset':       len(common_syn) > 0,
        'common_synset':     ' | '.join([synsets[i]['title'] for i in common_syn if i in synsets]),
        'has_common_hyp':    len(common_hyp) > 0,
        'common_hypernyms':  ' | '.join([t for _, t in common_hyp[:3]]),
    })

df_result2 = pd.DataFrame(results2)

print('\n══════════════════════════════════════════')
print(f'Всего пар: {len(df_result2)}')
print(f'Соседей в RuWordNet:        {df_result2["nb_in_rwn"].sum()} ({df_result2["nb_in_rwn"].mean()*100:.1f}%)')
print(f'В одном синсете:            {df_result2["same_synset"].sum()} ({df_result2["same_synset"].mean()*100:.1f}%)')
print(f'Общий гипероним (глуб. 3):  {df_result2["has_common_hyp"].sum()} ({df_result2["has_common_hyp"].mean()*100:.1f}%)')
print('══════════════════════════════════════════')

print('\n── По классам ──')
for cls in ['depression', 'anxiety', 'neutral']:
    sub = df_result2[df_result2['class'] == cls]
    print(f'{cls:<12}: синсет {sub["same_synset"].mean()*100:.1f}% | '
          f'гипероним {sub["has_common_hyp"].mean()*100:.1f}%')

print('\n── По ключевым словам ──')
for kw in df_result2['keyword'].unique():
    sub = df_result2[df_result2['keyword'] == kw]
    print(f'«{kw}»: синсет {sub["same_synset"].mean()*100:.1f}% | '
          f'гипероним {sub["has_common_hyp"].mean()*100:.1f}%')

# Примеры найденных связей
print('\n── Примеры пар с общим гиперонимом ──')
examples = df_result2[df_result2['has_common_hyp']].head(15)
for _, r in examples.iterrows():
    print(f'  «{r["keyword"]}» → «{r["neighbor"]}» [{r["class"]}]')
    print(f'    Общий гипероним: {r["common_hypernyms"]}')
    print()

Загружаем отношения...
Синсетов с гиперонимами: 18351
Анализируем гиперонимы...

══════════════════════════════════════════
Всего пар: 260
Соседей в RuWordNet:        213 (81.9%)
В одном синсете:            3 (1.2%)
Общий гипероним (глуб. 3):  18 (6.9%)
══════════════════════════════════════════

── По классам ──
depression  : синсет 2.2% | гипероним 5.6%
anxiety     : синсет 1.1% | гипероним 10.0%
neutral     : синсет 0.0% | гипероним 5.0%

── По ключевым словам ──
«депрессия»: синсет 0.0% | гипероним 3.3%
«тревога»: синсет 0.0% | гипероним 0.0%
«страх»: синсет 0.0% | гипероним 3.3%
«одиночество»: синсет 0.0% | гипероним 0.0%
«помощь»: синсет 3.3% | гипероним 6.7%
«лечение»: синсет 0.0% | гипероним 0.0%
«чувство»: синсет 3.3% | гипероним 36.7%
«жизнь»: синсет 3.3% | гипероним 10.0%
«плохо»: синсет 0.0% | гипероним 0.0%

── Примеры пар с общим гиперонимом ──
  «депрессия» → «болячка» [anxiety]
    Общий гипероним: ДЕПРЕССИВНОЕ РАССТРОЙСТВО

  «страх» → «ужас» [anxiety]
    Общий гиперо

In [7]:
# Перезапускаем с глубиной 6
results3 = []

for _, row in df_words.iterrows():
    keyword  = row['keyword']
    neighbor = row['neighbor']
    cls      = row['class']
    score    = row['score']

    kw_synsets_info = get_synset_info(keyword)
    nb_synsets_info = get_synset_info(neighbor)
    common_syn      = check_same_synset(keyword, neighbor)
    common_hyp      = find_common_hypernyms(keyword, neighbor, depth=6)  # ← глубина 6

    results3.append({
        'keyword':          keyword,
        'class':            cls,
        'neighbor':         neighbor,
        'score':            score,
        'kw_in_rwn':        len(kw_synsets_info) > 0,
        'nb_in_rwn':        len(nb_synsets_info) > 0,
        'kw_synsets':       ' | '.join([s['synset_title'] for s in kw_synsets_info]),
        'nb_synsets':       ' | '.join([s['synset_title'] for s in nb_synsets_info]),
        'same_synset':      len(common_syn) > 0,
        'common_synset':    ' | '.join([synsets[i]['title'] for i in common_syn if i in synsets]),
        'has_common_hyp':   len(common_hyp) > 0,
        'common_hypernyms': ' | '.join([t for _, t in common_hyp[:5]]),
    })

df_result3 = pd.DataFrame(results3)

print('══════════════════════════════════════════')
print(f'Глубина поиска: 6')
print(f'Всего пар: {len(df_result3)}')
print(f'В одном синсете:           {df_result3["same_synset"].sum()} ({df_result3["same_synset"].mean()*100:.1f}%)')
print(f'Общий гипероним (глуб. 6): {df_result3["has_common_hyp"].sum()} ({df_result3["has_common_hyp"].mean()*100:.1f}%)')
print('══════════════════════════════════════════')

print('\n── По ключевым словам ──')
for kw in df_result3['keyword'].unique():
    sub = df_result3[df_result3['keyword'] == kw]
    print(f'«{kw}»: синсет {sub["same_synset"].mean()*100:.1f}% | '
          f'гипероним {sub["has_common_hyp"].mean()*100:.1f}%')

══════════════════════════════════════════
Глубина поиска: 6
Всего пар: 260
В одном синсете:           3 (1.2%)
Общий гипероним (глуб. 6): 19 (7.3%)
══════════════════════════════════════════

── По ключевым словам ──
«депрессия»: синсет 0.0% | гипероним 6.7%
«тревога»: синсет 0.0% | гипероним 0.0%
«страх»: синсет 0.0% | гипероним 3.3%
«одиночество»: синсет 0.0% | гипероним 0.0%
«помощь»: синсет 3.3% | гипероним 6.7%
«лечение»: синсет 0.0% | гипероним 0.0%
«чувство»: синсет 3.3% | гипероним 36.7%
«жизнь»: синсет 3.3% | гипероним 10.0%
«плохо»: синсет 0.0% | гипероним 0.0%


In [8]:
# Сохраняем финальный файл с глубиной 6
wb_final = openpyxl.Workbook()

fill_dep = PatternFill(start_color='FFCCCC', fill_type='solid')
fill_anx = PatternFill(start_color='CCE5FF', fill_type='solid')
fill_neu = PatternFill(start_color='E0E0E0', fill_type='solid')
fill_yes = PatternFill(start_color='C6EFCE', fill_type='solid')
fill_hdr = PatternFill(start_color='D6E4F0', fill_type='solid')
fills_cls = {'depression': fill_dep, 'anxiety': fill_anx, 'neutral': fill_neu}

# ── Лист 1: Полный анализ ─────────────────────────────────────────
ws1 = wb_final.active
ws1.title = 'Полный анализ'
headers = ['Ключевое слово', 'Класс', 'Сосед (W2V)', 'Сходство',
           'Найден в RWN', 'Синсеты ключ. слова', 'Синсеты соседа',
           'Один синсет', 'Общий синсет', 'Общий гипероним', 'Гиперонимы']
ws1.append(headers)
for cell in ws1[1]:
    cell.font = Font(bold=True)
    cell.fill = fill_hdr

for _, row in df_result3.iterrows():
    ws1.append([
        row['keyword'], row['class'], row['neighbor'], row['score'],
        'Да' if row['nb_in_rwn'] else 'Нет',
        row['kw_synsets'], row['nb_synsets'],
        'Да' if row['same_synset'] else 'Нет',
        row['common_synset'],
        'Да' if row['has_common_hyp'] else 'Нет',
        row['common_hypernyms']
    ])
    lr = ws1.max_row
    ws1.cell(lr, 2).fill = fills_cls.get(row['class'], fill_neu)
    if row['same_synset'] or row['has_common_hyp']:
        for col in range(8, 12):
            ws1.cell(lr, col).fill = fill_yes

for i in range(1, ws1.max_column + 1):
    ws1.column_dimensions[get_column_letter(i)].width = 22

# ── Лист 2: Статистика ────────────────────────────────────────────
ws2 = wb_final.create_sheet('Статистика')
ws2.append(['Метрика', 'Depression', 'Anxiety', 'Neutral', 'Всего'])
for cell in ws2[1]:
    cell.font = Font(bold=True)
    cell.fill = fill_hdr

for metric_name, col in [
    ('Всего пар', None),
    ('Найдено в RuWordNet (%)', 'nb_in_rwn'),
    ('В одном синсете (%)', 'same_synset'),
    ('Общий гипероним глуб.6 (%)', 'has_common_hyp'),
    ('Семантически связаны (%)', None),
]:
    row_data = [metric_name]
    for cls in ['depression', 'anxiety', 'neutral']:
        sub = df_result3[df_result3['class'] == cls]
        if col is None and metric_name == 'Всего пар':
            row_data.append(len(sub))
        elif col is None:
            linked = sub['same_synset'] | sub['has_common_hyp']
            row_data.append(round(linked.mean() * 100, 1))
        else:
            row_data.append(round(sub[col].mean() * 100, 1))
    if col is None and metric_name == 'Всего пар':
        row_data.append(len(df_result3))
    elif col is None:
        linked = df_result3['same_synset'] | df_result3['has_common_hyp']
        row_data.append(round(linked.mean() * 100, 1))
    else:
        row_data.append(round(df_result3[col].mean() * 100, 1))
    ws2.append(row_data)

ws2.append([])
ws2.append(['Ключевое слово', 'Всего пар', 'В RuWordNet (%)',
            'Один синсет (%)', 'Гипероним (%)', 'Итого связаны (%)'])
for cell in ws2[ws2.max_row]:
    cell.font = Font(bold=True)
    cell.fill = fill_hdr

for kw in df_result3['keyword'].unique():
    sub = df_result3[df_result3['keyword'] == kw]
    linked = (sub['same_synset'] | sub['has_common_hyp']).mean() * 100
    ws2.append([
        kw, len(sub),
        round(sub['nb_in_rwn'].mean() * 100, 1),
        round(sub['same_synset'].mean() * 100, 1),
        round(sub['has_common_hyp'].mean() * 100, 1),
        round(linked, 1)
    ])

for i in range(1, ws2.max_column + 1):
    ws2.column_dimensions[get_column_letter(i)].width = 25

# ── Лист 3: Найденные связи ───────────────────────────────────────
ws3 = wb_final.create_sheet('Найденные связи')
ws3.append(['Ключевое слово', 'Класс', 'Сосед', 'Сходство',
            'Тип связи', 'Общий синсет / гипероним'])
for cell in ws3[1]:
    cell.font = Font(bold=True)
    cell.fill = fill_hdr

for _, row in df_result3[df_result3['same_synset'] | df_result3['has_common_hyp']].iterrows():
    link_type = 'Один синсет' if row['same_synset'] else 'Общий гипероним'
    link_name = row['common_synset'] if row['same_synset'] else row['common_hypernyms']
    ws3.append([
        row['keyword'], row['class'], row['neighbor'],
        row['score'], link_type, link_name
    ])
    lr = ws3.max_row
    ws3.cell(lr, 2).fill = fills_cls.get(row['class'], fill_neu)
    ws3.cell(lr, 5).fill = fill_yes

for i in range(1, ws3.max_column + 1):
    ws3.column_dimensions[get_column_letter(i)].width = 25

# ── Лист 4: Слова не найденные в RuWordNet ────────────────────────
ws4 = wb_final.create_sheet('Не в RuWordNet')
ws4.append(['Ключевое слово', 'Класс', 'Сосед', 'Сходство'])
for cell in ws4[1]:
    cell.font = Font(bold=True)
    cell.fill = fill_hdr

for _, row in df_result3[~df_result3['nb_in_rwn']].iterrows():
    ws4.append([row['keyword'], row['class'], row['neighbor'], row['score']])
    ws4.cell(ws4.max_row, 2).fill = fills_cls.get(row['class'], fill_neu)

for i in range(1, ws4.max_column + 1):
    ws4.column_dimensions[get_column_letter(i)].width = 22

OUT = '/content/drive/MyDrive/SFU 4/VKR/5. ruwordnet/ruwordnet_analysis_depth6.xlsx'
wb_final.save(OUT)
print('Сохранено: ruwordnet_analysis_depth6.xlsx')
print(f'Листы: Полный анализ | Статистика | Найденные связи | Не в RuWordNet')

# Итоговая сводка
linked_total = (df_result3['same_synset'] | df_result3['has_common_hyp']).sum()
print(f'\nИтого семантически связанных пар: {linked_total} ({linked_total/len(df_result3)*100:.1f}%)')
print(f'Не найдено в RuWordNet: {(~df_result3["nb_in_rwn"]).sum()} слов')

Сохранено: ruwordnet_analysis_depth6.xlsx
Листы: Полный анализ | Статистика | Найденные связи | Не в RuWordNet

Итого семантически связанных пар: 19 (7.3%)
Не найдено в RuWordNet: 47 слов


# **АНАЛИЗ 1: Семантическое расстояние через путь в графе RuWordNet**

In [9]:
# ══════════════════════════════════════════════════════════════════
# АНАЛИЗ 1: Семантическое расстояние через путь в графе RuWordNet
# ══════════════════════════════════════════════════════════════════

from collections import deque

def shortest_path(synset_id1, synset_id2, max_depth=8):
    """BFS поиск кратчайшего пути между двумя синсетами через гиперонимы"""
    if synset_id1 == synset_id2:
        return 0

    visited = {synset_id1}
    queue = deque([(synset_id1, 0)])

    while queue:
        current, depth = queue.popleft()
        if depth >= max_depth:
            continue
        # Идём вверх (гиперонимы) и вниз (гипонимы)
        neighbors = hypernyms.get(current, []) + hyponyms.get(current, [])
        for neighbor in neighbors:
            if neighbor == synset_id2:
                return depth + 1
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, depth + 1))
    return None  # путь не найден

def semantic_distance(word1, word2, max_depth=8):
    """Минимальное расстояние между всеми синсетами двух слов"""
    s1_ids = word_to_synsets.get(word1, [])
    s2_ids = word_to_synsets.get(word2, [])

    if not s1_ids or not s2_ids:
        return None

    min_dist = None
    best_path = None
    for sid1 in s1_ids:
        for sid2 in s2_ids:
            dist = shortest_path(sid1, sid2, max_depth)
            if dist is not None:
                if min_dist is None or dist < min_dist:
                    min_dist = dist
                    best_path = (synsets[sid1]['title'], synsets[sid2]['title'])
    return min_dist, best_path

# Применяем к датафрейму
print('Считаем семантические расстояния...')
distances = []
for _, row in df_words.iterrows():
    result = semantic_distance(row['keyword'], row['neighbor'])
    if result and result[0] is not None:
        dist, path = result
        distances.append({
            'keyword':   row['keyword'],
            'class':     row['class'],
            'neighbor':  row['neighbor'],
            'score':     row['score'],
            'distance':  dist,
            'path':      f'{path[0]} → {path[1]}' if path else ''
        })
    else:
        distances.append({
            'keyword':  row['keyword'],
            'class':    row['class'],
            'neighbor': row['neighbor'],
            'score':    row['score'],
            'distance': None,
            'path':     'не найден'
        })

df_dist = pd.DataFrame(distances)

print('\n── Среднее расстояние по классам ──')
for cls in ['depression', 'anxiety', 'neutral']:
    sub = df_dist[(df_dist['class'] == cls) & df_dist['distance'].notna()]
    if len(sub) > 0:
        print(f'{cls:<12}: среднее={sub["distance"].mean():.2f} | '
              f'найдено путей={len(sub)} из {len(df_dist[df_dist["class"]==cls])}')

print('\n── Среднее расстояние по ключевым словам ──')
for kw in df_dist['keyword'].unique():
    sub = df_dist[(df_dist['keyword'] == kw) & df_dist['distance'].notna()]
    if len(sub) > 0:
        print(f'«{kw}»: среднее расстояние={sub["distance"].mean():.2f} '
              f'({len(sub)} пар с найденным путём)')

print('\n── Топ-10 ближайших пар ──')
closest = df_dist[df_dist['distance'].notna()].nsmallest(10, 'distance')
for _, r in closest.iterrows():
    print(f'  [{r["class"]}] «{r["keyword"]}» → «{r["neighbor"]}» '
          f'расстояние={r["distance"]} | {r["path"]}')

Считаем семантические расстояния...

── Среднее расстояние по классам ──
depression  : среднее=4.79 | найдено путей=34 из 90
anxiety     : среднее=4.69 | найдено путей=39 из 90
neutral     : среднее=5.17 | найдено путей=29 из 80

── Среднее расстояние по ключевым словам ──
«депрессия»: среднее расстояние=4.93 (15 пар с найденным путём)
«тревога»: среднее расстояние=6.00 (14 пар с найденным путём)
«страх»: среднее расстояние=5.67 (18 пар с найденным путём)
«одиночество»: среднее расстояние=6.38 (8 пар с найденным путём)
«помощь»: среднее расстояние=4.45 (11 пар с найденным путём)
«лечение»: среднее расстояние=4.86 (7 пар с найденным путём)
«чувство»: среднее расстояние=2.78 (18 пар с найденным путём)
«жизнь»: среднее расстояние=4.73 (11 пар с найденным путём)

── Топ-10 ближайших пар ──
  [anxiety] «помощь» → «поддержка» расстояние=0.0 | ПОМОЩЬ, ПОДДЕРЖКА → ПОМОЩЬ, ПОДДЕРЖКА
  [depression] «чувство» → «ощущение» расстояние=0.0 | ЧУВСТВО, ЭМОЦИЯ → ЧУВСТВО, ЭМОЦИЯ
  [depression] «жизнь» →

# **АНАЛИЗ 2: Семантические категории соседей (верхнеуровневые гиперонимы)**

In [10]:
# ══════════════════════════════════════════════════════════════════
# АНАЛИЗ 2: Семантические категории соседей (верхнеуровневые гиперонимы)
# ══════════════════════════════════════════════════════════════════

def get_top_hypernym(synset_id, max_depth=8):
    """Поднимается по иерархии до корневого гиперонима"""
    current = synset_id
    path = [synsets[current]['title']] if current in synsets else []

    for _ in range(max_depth):
        parents = hypernyms.get(current, [])
        if not parents:
            break
        current = parents[0]
        if current in synsets:
            path.append(synsets[current]['title'])

    return path[-1] if path else None, path

def get_level2_hypernym(synset_id, target_depth=2):
    """Получает гипероним на уровне 2 от слова — семантическая категория"""
    current = synset_id
    for _ in range(target_depth):
        parents = hypernyms.get(current, [])
        if not parents:
            break
        current = parents[0]
    return synsets.get(current, {}).get('title', None)

# Для каждого соседа определяем его семантическую категорию
print('Определяем семантические категории...')
cat_results = []

for _, row in df_words.iterrows():
    neighbor = row['neighbor']
    sids = word_to_synsets.get(neighbor, [])

    if not sids:
        cat_results.append({
            'keyword':    row['keyword'],
            'class':      row['class'],
            'neighbor':   neighbor,
            'score':      row['score'],
            'synset':     'не найден',
            'category_2': 'не найден',
            'top_category': 'не найден'
        })
        continue

    # Берём первый синсет
    sid = sids[0]
    synset_title = synsets[sid]['title']
    cat2 = get_level2_hypernym(sid, target_depth=2)
    top, _ = get_top_hypernym(sid)

    cat_results.append({
        'keyword':      row['keyword'],
        'class':        row['class'],
        'neighbor':     neighbor,
        'score':        row['score'],
        'synset':       synset_title,
        'category_2':   cat2 or synset_title,
        'top_category': top or synset_title
    })

df_cat = pd.DataFrame(cat_results)

# Топ категорий по классам
print('\n══════════════════════════════════════════')
print('Топ-7 семантических категорий по классам')
print('(уровень 2 от слова в иерархии RuWordNet)')
print('══════════════════════════════════════════')

for cls in ['depression', 'anxiety', 'neutral']:
    sub = df_cat[
        (df_cat['class'] == cls) &
        (df_cat['category_2'] != 'не найден')
    ]
    print(f'\n── {cls} ──')
    top_cats = sub['category_2'].value_counts().head(7)
    for cat, count in top_cats.items():
        pct = count / len(sub) * 100
        # Примеры слов в этой категории
        examples = sub[sub['category_2'] == cat]['neighbor'].tolist()[:3]
        print(f'  {cat:<40} {count:>3} ({pct:.0f}%) — {", ".join(examples)}')

# Уникальные категории для каждого класса
print('\n── Уникальные категории (только в одном классе) ──')
cats_dep = set(df_cat[df_cat['class']=='depression']['category_2'])
cats_anx = set(df_cat[df_cat['class']=='anxiety']['category_2'])
cats_neu = set(df_cat[df_cat['class']=='neutral']['category_2'])

unique_dep = cats_dep - cats_anx - cats_neu - {'не найден'}
unique_anx = cats_anx - cats_dep - cats_neu - {'не найден'}
unique_neu = cats_neu - cats_dep - cats_anx - {'не найден'}

print(f'\nТолько в depression: {sorted(unique_dep)}')
print(f'Только в anxiety:    {sorted(unique_anx)}')
print(f'Только в neutral:    {sorted(unique_neu)}')

Определяем семантические категории...

══════════════════════════════════════════
Топ-7 семантических категорий по классам
(уровень 2 от слова в иерархии RuWordNet)
══════════════════════════════════════════

── depression ──
  ЗАЦИКЛИТЬСЯ НА ОДНОМ И ТОМ ЖЕ              3 (4%) — обращение, обращаться, обратиться
  БЕЗНАДЕЖНЫЙ (НЕИСПРАВИМЫЙ)                 3 (4%) — безысходность, непоправимый, безысходность
  СОВМЕСТИТЬ                                 2 (3%) — сочетать, сочетание
  ДЕВАТЬ НЕИЗВЕСТНО КУДА                     2 (3%) — девать, терять
  ИСКЛЮЧИТЬ ВОЗМОЖНОСТЬ                      1 (1%) — исключать
  ПОВЫШАТЕЛЬНЫЙ ТРЕНД                        1 (1%) — динамика
  БРАВЫЙ, МОЛОДЦЕВАТЫЙ                       1 (1%) — лихой

── anxiety ──
  ПОДВЕРЖЕННОСТЬ                             3 (4%) — склонность, подверженный, условие
  НЕОПРЕДЕЛЕННЫЙ (ТОЧНО НЕ УСТАНОВЛЕННЫЙ)    2 (3%) — неопределенность, неизвестный
  ЗАЦИКЛИТЬСЯ НА ОДНОМ И ТОМ ЖЕ              2 (3%) — обращение, обратит

# **АНАЛИЗ 3: Валидация ПОЛНОЙ онтологии через RuWordNet**

In [11]:
# ══════════════════════════════════════════════════════════════════
# АНАЛИЗ 3: Валидация ПОЛНОЙ онтологии через RuWordNet
# ══════════════════════════════════════════════════════════════════

# Полная онтология — все уникальные леммы из всех категорий
ONTOLOGY_DEP_FULL = list(set([
    # depression_keywords
    'депрессия', 'грусть', 'печаль', 'тоска', 'уныние', 'безнадежность',
    'беспомощность', 'опустошенность', 'апатия', 'безразличие', 'одиночество',
    'подавленность', 'угнетение', 'нерешительность', 'неуверенность',
    'забывчивость', 'вина', 'суицид', 'смерть', 'усталость', 'бессонница',
    'слабость', 'изоляция', 'пустота', 'безысходность',
    # negative_emotion_words — депрессивная лексика
    'никчемность', 'ничтожность', 'бессмысленность', 'отчаяние',
    'обреченность', 'беспросветность',
    # extra_depression
    'антидепрессант', 'психиатр', 'психолог', 'терапевт', 'психотерапевт',
    'лечение', 'диагноз', 'расстройство', 'клиника', 'стационар',
    'больница', 'препарат', 'выгорание', 'истощение', 'ангедония',
    'срыв', 'плакать', 'рыдать', 'тяжело',
]))

ONTOLOGY_ANX_FULL = list(set([
    # anxiety_keywords
    'тревога', 'беспокойство', 'волнение', 'страх', 'испуг', 'паника',
    'нервозность', 'напряжение', 'опасение', 'раздражительность',
    'сердцебиение', 'дрожь', 'тремор', 'одышка', 'головокружение',
    'тошнота', 'избегание', 'ритуал', 'тревожность',
    # negative_emotion_words — тревожная лексика
    'скованность', 'зажатость', 'сомнение', 'мнительность', 'подозрительность',
    # extra_anxiety
    'транквилизатор', 'приступ', 'атака', 'нервничать', 'волноваться',
    'переживать', 'бояться', 'накручивать', 'прокручивать',
    'избегать', 'перепроверять', 'контролировать', 'трястись',
    'потеть', 'задыхаться',
]))

# Общие категории (absolutist, pronouns, certainty, swear — для обоих классов)
ONTOLOGY_COMMON = list(set([
    # absolutist
    'всегда', 'никогда', 'постоянно', 'вечно', 'навсегда',
    'полностью', 'абсолютно', 'совершенно', 'невозможно',
    'безнадежно', 'бесполезно', 'бессмысленно', 'неизбежно',
    'ужасно', 'кошмарно', 'жутко', 'страшно',
    # negative_emotion общие
    'скорбь', 'горе', 'ужас', 'боязнь', 'трепет',
    'гнев', 'злость', 'ярость', 'раздражение', 'ненависть',
    'отвращение', 'презрение', 'неприязнь', 'стыд', 'позор',
    'раскаяние', 'сожаление',
]))

print(f'Слов в словаре депрессии: {len(ONTOLOGY_DEP_FULL)}')
print(f'Слов в словаре тревоги:   {len(ONTOLOGY_ANX_FULL)}')
print(f'Общих слов:               {len(ONTOLOGY_COMMON)}')

def check_ontology_in_rwn(word_list, label):
    in_rwn, not_rwn, results = [], [], []
    for word in sorted(word_list):
        sids = word_to_synsets.get(word, [])
        if sids:
            titles = [synsets[s]['title'] for s in sids[:2]]
            in_rwn.append(word)
            results.append({'word': word, 'found': True,
                           'synsets': ' | '.join(titles)})
        else:
            not_rwn.append(word)
            results.append({'word': word, 'found': False, 'synsets': '—'})
    print(f'\n── Словарь {label} ({len(word_list)} слов) ──')
    for r in results:
        mark = '✓' if r['found'] else '✗'
        print(f'  {mark} {r["word"]:<25} → {r["synsets"][:70]}')
    print(f'\nНайдено: {len(in_rwn)}/{len(word_list)} ({len(in_rwn)/len(word_list)*100:.0f}%)')
    print(f'Не найдено: {sorted(not_rwn)}')
    return in_rwn, not_rwn

dep_in_rwn, dep_not_rwn = check_ontology_in_rwn(ONTOLOGY_DEP_FULL, 'ДЕПРЕССИИ')
anx_in_rwn, anx_not_rwn = check_ontology_in_rwn(ONTOLOGY_ANX_FULL, 'ТРЕВОГИ')
com_in_rwn, com_not_rwn = check_ontology_in_rwn(ONTOLOGY_COMMON, 'ОБЩИХ МАРКЕРОВ')

# Прямые пересечения синсетов
print('\n══ Прямые пересечения синсетов (депрессия ∩ тревога) ══')
shared = []
for w_dep in dep_in_rwn:
    for w_anx in anx_in_rwn:
        common = check_same_synset(w_dep, w_anx)
        if common:
            for sid in common:
                shared.append({
                    'dep_word': w_dep, 'anx_word': w_anx,
                    'synset': synsets[sid]['title'],
                    'synset_words': synsets[sid]['words'][:6]
                })

if shared:
    print(f'Найдено {len(shared)} пересечений:')
    for s in shared:
        print(f'  {s["dep_word"]:<20} + {s["anx_word"]:<20} → [{s["synset"]}]')
        print(f'    Слова синсета: {s["synset_words"]}')
else:
    print('Прямых пересечений не найдено')

# Близкие пары d≤2
print('\n══ Семантически близкие пары из разных словарей (d≤2) ══')
close_pairs = []
for w_dep in dep_in_rwn:
    for w_anx in anx_in_rwn:
        result = semantic_distance(w_dep, w_anx, max_depth=4)
        if result and result[0] is not None and result[0] <= 2:
            close_pairs.append({
                'dep_word': w_dep, 'anx_word': w_anx,
                'distance': result[0],
                'path': f'{result[1][0]} → {result[1][1]}' if result[1] else ''
            })

close_pairs.sort(key=lambda x: (x['distance'], x['dep_word']))
print(f'Найдено близких пар (d≤2): {len(close_pairs)}')
for p in close_pairs:
    print(f'  d={p["distance"]} | {p["dep_word"]:<20} ↔ {p["anx_word"]:<20} | {p["path"]}')

# Сохраняем для Excel
df_ont_dep = pd.DataFrame([{
    'слово': w, 'класс': 'depression',
    'найдено_в_RWN': w in dep_in_rwn,
    'синсеты': ' | '.join([synsets[s]['title']
                for s in word_to_synsets.get(w, [])[:2]])
} for w in ONTOLOGY_DEP_FULL])

df_ont_anx = pd.DataFrame([{
    'слово': w, 'класс': 'anxiety',
    'найдено_в_RWN': w in anx_in_rwn,
    'синсеты': ' | '.join([synsets[s]['title']
                for s in word_to_synsets.get(w, [])[:2]])
} for w in ONTOLOGY_ANX_FULL])

df_ont_com = pd.DataFrame([{
    'слово': w, 'класс': 'common',
    'найдено_в_RWN': w in com_in_rwn,
    'синсеты': ' | '.join([synsets[s]['title']
                for s in word_to_synsets.get(w, [])[:2]])
} for w in ONTOLOGY_COMMON])

df_ont_all  = pd.concat([df_ont_dep, df_ont_anx, df_ont_com], ignore_index=True)
df_close_pairs = pd.DataFrame(close_pairs)
print(f'\nВсего строк в таблице онтологии: {len(df_ont_all)}')

Слов в словаре депрессии: 50
Слов в словаре тревоги:   39
Общих слов:               34

── Словарь ДЕПРЕССИИ (50 слов) ──
  ✗ ангедония                 → —
  ✓ антидепрессант            → АНТИДЕПРЕССАНТ
  ✓ апатия                    → АПАТИЯ
  ✓ безнадежность             → БЕЗНАДЕЖНЫЙ, ОБРЕЧЕННЫЙ НА НЕУДАЧУ | БЕЗНАДЕЖНЫЙ (НЕИСПРАВИМЫЙ)
  ✓ безразличие               → РАВНОДУШНЫЙ, БЕЗРАЗЛИЧНЫЙ
  ✓ безысходность             → БЕЗНАДЕЖНЫЙ (НЕИСПРАВИМЫЙ)
  ✓ беспомощность             → БЕСПОМОЩНЫЙ (ОЧЕНЬ ПЛОХОЙ, СЛАБЫЙ, БЕЗДАРНЫЙ) | БЕСПОМОЩНЫЙ (НУЖДАЮЩИЙ
  ✓ беспросветность           → МРАЧНЫЙ, БЕСПРОСВЕТНЫЙ
  ✓ бессмысленность           → БЕЗРЕЗУЛЬТАТНЫЙ | БЕССМЫСЛЕННЫЙ
  ✓ бессонница                → БЕССОННИЦА
  ✓ больница                  → БОЛЬНИЦА
  ✓ вина                      → ВИНА | ВИНА (ПРИЧИНА НЕБЛАГОПРИЯТНОГО)
  ✓ выгорание                 → ВЫЦВЕСТИ НА СОЛНЦЕ
  ✓ грусть                    → ГРУСТЬ
  ✓ депрессия                 → ЭКОНОМИЧЕСКАЯ ДЕПРЕССИЯ | ДЕПРЕССИВНОЕ РАССТРО

# **АНАЛИЗ 4: Сравнение синсетов ключевых слов с результатами Word2Vec**

In [12]:
# ══════════════════════════════════════════════════════════════════
# АНАЛИЗ 4: Сравнение синсетов ключевых слов с результатами Word2Vec
# Берём все слова из синсетов ДЕПРЕССИВНОЕ РАССТРОЙСТВО и
# ТРЕВОГА, БЕСПОКОЙСТВО — и сравниваем с соседями Word2Vec
# ══════════════════════════════════════════════════════════════════

print('══════════════════════════════════════════════════════')
print('АНАЛИЗ 4: Синсеты RuWordNet vs соседи Word2Vec')
print('══════════════════════════════════════════════════════')

# Находим все релевантные синсеты
TARGET_SYNSETS = {
    'депрессия': ['ДЕПРЕССИВНОЕ РАССТРОЙСТВО', 'ПОДАВЛЕННОЕ СОСТОЯНИЕ',
                  'АПАТИЯ', 'ТОСКА (ГРУСТЬ, УНЫНИЕ)'],
    'тревога':   ['ТРЕВОГА, БЕСПОКОЙСТВО', 'СТРАХ, БОЯЗНЬ',
                  'ПАНИКА', 'НЕРВНОЕ СОСТОЯНИЕ'],
}

# Собираем все слова из целевых синсетов
rwn_dep_words = set()
rwn_anx_words = set()

for sid, s in synsets.items():
    title = s['title']
    if any(t in title for t in TARGET_SYNSETS['депрессия']):
        for w in s['words']:
            rwn_dep_words.add(w.lower().split()[0])  # берём первое слово леммы
    if any(t in title for t in TARGET_SYNSETS['тревога']):
        for w in s['words']:
            rwn_anx_words.add(w.lower().split()[0])

print(f'\nСлов в синсетах депрессии RuWordNet: {len(rwn_dep_words)}')
print(f'Слова: {sorted(rwn_dep_words)}')
print(f'\nСлов в синсетах тревоги RuWordNet: {len(rwn_anx_words)}')
print(f'Слова: {sorted(rwn_anx_words)}')

# Сравниваем с соседями Word2Vec
print('\n── Соседи Word2Vec которые есть в синсетах RuWordNet ──')
w2v_neighbors_all = df_words['neighbor'].str.lower().unique()

matched_dep = set(w2v_neighbors_all) & rwn_dep_words
matched_anx = set(w2v_neighbors_all) & rwn_anx_words
matched_both = matched_dep & matched_anx

print(f'\nСоседей W2V совпавших с синсетами депрессии RuWordNet: {len(matched_dep)}')
for w in sorted(matched_dep):
    # В каком классе W2V встречается это слово?
    classes = df_words[df_words['neighbor'].str.lower()==w]['class'].tolist()
    kws = df_words[df_words['neighbor'].str.lower()==w]['keyword'].tolist()
    print(f'  ✓ {w:<20} → встречается в: {list(zip(kws, classes))}')

print(f'\nСоседей W2V совпавших с синсетами тревоги RuWordNet: {len(matched_anx)}')
for w in sorted(matched_anx):
    classes = df_words[df_words['neighbor'].str.lower()==w]['class'].tolist()
    kws = df_words[df_words['neighbor'].str.lower()==w]['keyword'].tolist()
    print(f'  ✓ {w:<20} → встречается в: {list(zip(kws, classes))}')

print(f'\nСовпадают с обоими: {sorted(matched_both)}')

# Финальная сводная таблица всех 4 анализов
print('\n══════════════════════════════════════════════════════')
print('ИТОГОВАЯ СВОДКА ВСЕХ 4 АНАЛИЗОВ')
print('══════════════════════════════════════════════════════')
print(f'Анализ 1 (семант. расстояние):')
print(f'  Найдено путей: {df_dist["distance"].notna().sum()} из {len(df_dist)}')
print(f'  Среднее расстояние: depression={df_dist[df_dist["class"]=="depression"]["distance"].mean():.2f} | '
      f'anxiety={df_dist[df_dist["class"]=="anxiety"]["distance"].mean():.2f} | '
      f'neutral={df_dist[df_dist["class"]=="neutral"]["distance"].mean():.2f}')
print(f'  Ближайшие пары (d=0): {(df_dist["distance"]==0).sum()}')
print(f'\nАнализ 2 (семант. категории):')
print(f'  Уникальных категорий depression: {len(unique_dep)}')
print(f'  Уникальных категорий anxiety: {len(unique_anx)}')
print(f'  Уникальных категорий neutral: {len(unique_neu)}')
print(f'\nАнализ 3 (валидация онтологии):')
print(f'  Депрессия в RuWordNet: 48/50 (96%)')
print(f'  Тревога в RuWordNet: 38/39 (97%)')
print(f'  Общие маркеры в RuWordNet: 17/34 (50%)')
print(f'  Прямых пересечений синсетов: {len(shared)}')
print(f'  Близких пар (d≤2): {len(close_pairs)}')
print(f'\nАнализ 4 (синсеты vs W2V):')
print(f'  Соседей W2V в синсетах депрессии RuWordNet: {len(matched_dep)}')
print(f'  Соседей W2V в синсетах тревоги RuWordNet: {len(matched_anx)}')

══════════════════════════════════════════════════════
АНАЛИЗ 4: Синсеты RuWordNet vs соседи Word2Vec
══════════════════════════════════════════════════════

Слов в синсетах депрессии RuWordNet: 30
Слова: ['апатический', 'апатичность', 'апатичный', 'апатия', 'все', 'депрессивность', 'депрессивный', 'депрессия', 'душевный', 'камень', 'кручина', 'меланхолический', 'меланхолия', 'нервный', 'подавленность', 'подавленный', 'пришибленность', 'пришибленный', 'смертный', 'состояние', 'тоска', 'тоскливость', 'тоскливый', 'тяжесть', 'угнетенность', 'угнетенный', 'удрученность', 'удрученный', 'хандра', 'хандрить']

Слов в синсетах тревоги RuWordNet: 39
Слова: ['беспокойность', 'беспокойный', 'беспокойство', 'боязнь', 'весь', 'взвинченность', 'взвинченный', 'встревоженность', 'встревоженный', 'дергаться', 'задергаться', 'занервничать', 'изнервничаться', 'комок', 'на', 'нервишки', 'нервничать', 'нервность', 'нервный', 'нервозность', 'нервозный', 'неспокойный', 'обеспокоенность', 'обеспокоенный', 'п

# **EXCEL**

In [13]:
# ══════════════════════════════════════════════════════════════════
# ФИНАЛЬНЫЙ EXCEL: все 4 анализа в одном файле
# ══════════════════════════════════════════════════════════════════

wb_final = openpyxl.Workbook()

fill_dep = PatternFill(start_color='FFCCCC', fill_type='solid')
fill_anx = PatternFill(start_color='CCE5FF', fill_type='solid')
fill_neu = PatternFill(start_color='E0E0E0', fill_type='solid')
fill_yes = PatternFill(start_color='C6EFCE', fill_type='solid')
fill_hdr = PatternFill(start_color='D6E4F0', fill_type='solid')
fills_cls = {'depression': fill_dep, 'anxiety': fill_anx, 'neutral': fill_neu}

def add_header(ws, headers):
    ws.append(headers)
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.fill = fill_hdr
    for i in range(1, len(headers)+1):
        ws.column_dimensions[get_column_letter(i)].width = 25

# ── Лист 1: Итоговая сводка ───────────────────────────────────────
ws0 = wb_final.active
ws0.title = 'Сводка'
add_header(ws0, ['Анализ', 'Метрика', 'Значение', 'Интерпретация'])

summary_rows = [
    ['Анализ 1', 'Найдено путей в графе RuWordNet',
     f'{df_dist["distance"].notna().sum()} из {len(df_dist)} (37.7%)',
     'Семантические пути существуют для трети пар'],
    ['Анализ 1', 'Среднее расстояние depression',
     f'{df_dist[df_dist["class"]=="depression"]["distance"].mean():.2f}',
     'Умеренная семантическая близость'],
    ['Анализ 1', 'Среднее расстояние anxiety',
     f'{df_dist[df_dist["class"]=="anxiety"]["distance"].mean():.2f}',
     'Чуть ближе чем depression'],
    ['Анализ 1', 'Среднее расстояние neutral',
     f'{df_dist[df_dist["class"]=="neutral"]["distance"].mean():.2f}',
     'Наибольшее расстояние — менее связаны'],
    ['Анализ 1', 'Пар в одном синсете (d=0)', '3',
     'помощь-поддержка, чувство-ощущение, жизнь-существование'],
    ['Анализ 1', 'Слово «чувство» — среднее расстояние', '2.78',
     'Наименьшее — эмоциональный домен хорошо представлен'],
    ['', '', '', ''],
    ['Анализ 2', 'Уникальных категорий depression', '59',
     'Наиболее семантически разнообразный класс'],
    ['Анализ 2', 'Уникальных категорий anxiety', '53', ''],
    ['Анализ 2', 'Уникальных категорий neutral', '53', ''],
    ['Анализ 2', 'Общая категория всех 3 классов',
     'ЗАЦИКЛИТЬСЯ НА ОДНОМ И ТОМ ЖЕ',
     'Навязчивость как общий маркер расстройств'],
    ['', '', '', ''],
    ['Анализ 3', 'Депрессия в RuWordNet', '48/50 (96%)',
     'Онтология хорошо покрыта тезаурусом'],
    ['Анализ 3', 'Тревога в RuWordNet', '38/39 (97%)',
     'Онтология хорошо покрыта тезаурусом'],
    ['Анализ 3', 'Общие маркеры в RuWordNet', '17/34 (50%)',
     'Абсолютистские слова не в тезаурусе — разговорная лексика'],
    ['Анализ 3', 'Прямых пересечений синсетов', '1',
     'неуверенность ↔ сомнение → СОМНЕНИЕ, НЕУВЕРЕННОСТЬ'],
    ['Анализ 3', 'Близких пар (d≤2)', '17',
     'клиника-одышка, бессонница-тошнота, грусть-волнение...'],
    ['', '', '', ''],
    ['Анализ 4', 'Соседей W2V в синсетах депрессии RuWordNet', '1',
     'подавленность (в модели depression для слова тревога)'],
    ['Анализ 4', 'Соседей W2V в синсетах тревоги RuWordNet', '0',
     'W2V и RuWordNet используют разные принципы близости'],
]

for row in summary_rows:
    ws0.append(row)
    if row[0].startswith('Анализ'):
        ws0.cell(ws0.max_row, 1).font = Font(bold=True)

# ── Лист 2: Анализ 1 — семантические расстояния ──────────────────
ws1 = wb_final.create_sheet('1. Расстояния')
add_header(ws1, ['Ключевое слово', 'Класс', 'Сосед (W2V)',
                 'Сходство W2V', 'Расстояние RWN', 'Путь в RWN'])

for _, row in df_dist.iterrows():
    ws1.append([row['keyword'], row['class'], row['neighbor'],
                row['score'], row['distance'], row['path']])
    lr = ws1.max_row
    ws1.cell(lr, 2).fill = fills_cls.get(row['class'], fill_neu)
    if pd.notna(row['distance']) and row['distance'] <= 2:
        ws1.cell(lr, 5).fill = fill_yes

# ── Лист 3: Анализ 2 — категории ─────────────────────────────────
ws2 = wb_final.create_sheet('2. Категории')
add_header(ws2, ['Ключевое слово', 'Класс', 'Сосед',
                 'Сходство', 'Синсет', 'Категория (ур.2)', 'Верхн. категория'])

for _, row in df_cat.iterrows():
    ws2.append([row['keyword'], row['class'], row['neighbor'],
                row['score'], row['synset'],
                row['category_2'], row['top_category']])
    ws2.cell(ws2.max_row, 2).fill = fills_cls.get(row['class'], fill_neu)

# ── Лист 4: Анализ 3 — онтология ─────────────────────────────────
ws3 = wb_final.create_sheet('3. Онтология в RWN')
add_header(ws3, ['Слово', 'Класс', 'Найдено в RuWordNet', 'Синсеты'])

for _, row in df_ont_all.iterrows():
    ws3.append([row['слово'], row['класс'],
                'Да' if row['найдено_в_RWN'] else 'Нет', row['синсеты']])
    lr = ws3.max_row
    ws3.cell(lr, 2).fill = fills_cls.get(row['класс'], fill_neu)
    if row['найдено_в_RWN']:
        ws3.cell(lr, 3).fill = fill_yes

# ── Лист 5: Анализ 3 — близкие пары ──────────────────────────────
ws3b = wb_final.create_sheet('3. Близкие пары онтологии')
add_header(ws3b, ['Слово депрессии', 'Слово тревоги',
                  'Расстояние', 'Путь в RuWordNet'])

for p in close_pairs:
    ws3b.append([p['dep_word'], p['anx_word'], p['distance'], p['path']])
    lr = ws3b.max_row
    if p['distance'] == 0:
        ws3b.cell(lr, 3).fill = fill_yes
    elif p['distance'] == 1:
        ws3b.cell(lr, 3).fill = PatternFill(start_color='FFFACD', fill_type='solid')

# ── Лист 6: Анализ 4 — синсеты RWN ───────────────────────────────
ws4 = wb_final.create_sheet('4. Синсеты RWN vs W2V')

add_header(ws4, ['Категория', 'Слова из синсетов RuWordNet'])
ws4.append(['Синсеты ДЕПРЕССИИ', ', '.join(sorted(rwn_dep_words))])
ws4.append(['Синсеты ТРЕВОГИ',   ', '.join(sorted(rwn_anx_words))])
ws4.append([])
add_header(ws4, ['Совпадение W2V ↔ RWN', 'Слово', 'Ключ. слово W2V',
                 'Класс W2V', 'Синсет RuWordNet'])

for w in sorted(matched_dep):
    rows_w = df_words[df_words['neighbor'].str.lower() == w]
    for _, r in rows_w.iterrows():
        sids = word_to_synsets.get(w, [])
        title = synsets[sids[0]]['title'] if sids else '—'
        ws4.append(['Депрессия', w, r['keyword'], r['class'], title])
        ws4.cell(ws4.max_row, 4).fill = fills_cls.get(r['class'], fill_neu)

if not matched_dep and not matched_anx:
    ws4.append(['Совпадений не найдено', '', '', '', ''])

OUT = '/content/drive/MyDrive/SFU 4/VKR/5. ruwordnet/ruwordnet_full_analysis.xlsx'
wb_final.save(OUT)
print('Сохранено: ruwordnet_full_analysis.xlsx')
print('Листы: Сводка | 1.Расстояния | 2.Категории | '
      '3.Онтология в RWN | 3.Близкие пары | 4.Синсеты RWN vs W2V')

Сохранено: ruwordnet_full_analysis.xlsx
Листы: Сводка | 1.Расстояния | 2.Категории | 3.Онтология в RWN | 3.Близкие пары | 4.Синсеты RWN vs W2V
